## Step 1: install libraries

In [ ]:
!pip install numpy scikit-learn scipy matplotlib seaborn


## Step 2: Imports + Setup

In [ ]:
import numpy as np
import pandas as pd
import os
import sys
import scipy.io as sio
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.linear_model import Ridge

print("Initializing SSTM-IS Environment...")

# SEED IV constants
NUM_CLASSES = 4          # Neutral, Sad, Fear, Happy
CHANNELS = 62            # 62-channel EEG
BANDS = 5                # Delta, Theta, Alpha, Beta, Gamma
FEATURE_DIM = 310        # 62 * 5 = 310 features (DE)
NUM_SUBJECTS = 15        # 15 subjects

# Mind To Music Parameters
K_INSTANCES = 10000       #  Number of source instances to select
RHO = 0.1                #  Mean adjustment factor

# 0: Neutral, 1: Sad, 2: Fear, 3: Happy
SESSION_LABELS_FALLBACK = {
    1: [1, 2, 3, 0, 2, 0, 0, 1, 0, 1, 2, 1, 1, 1, 2, 3, 2, 2, 3, 3, 0, 3, 0, 3],
    2: [2, 1, 3, 0, 0, 2, 0, 2, 3, 3, 2, 3, 2, 0, 1, 1, 2, 1, 0, 3, 0, 1, 3, 1],
    3: [1, 2, 2, 1, 3, 3, 3, 1, 1, 2, 1, 0, 2, 3, 3, 0, 2, 3, 0, 0, 2, 0, 1, 0]
}

print("Imports loaded. SEED IV configuration set.")

## Step 3: Data Loading


In [ ]:
ROOT_PATH = 'eeg_feature_smooth'

def load_labels_from_file(root_path, session_id):
    #load sessionX_label.mat from the root path.

    import scipy.io as sio
    possible_names = [f"session{session_id}_label.mat", f"Session{session_id}_label.mat", f"label{session_id}.mat"]

    for name in possible_names:
        path = os.path.join(root_path, name)
        if os.path.exists(path):
            try:
                mat = sio.loadmat(path)
                for key in mat:
                    if 'label' in key and not key.startswith('__'):
                        labels = mat[key].flatten()
                        if np.min(labels) == 1: labels = labels - 1
                        print(f"  Loaded labels from {name}: {labels}")
                        return labels
            except Exception as e:
                print(f"  Found {name} but failed to load: {e}")
    return None

def load_seed4_from_folders(root_path):
    #iterates through eeg_feature_smooth/1, /2, /3.
    #returns: X, y, subjects, sessions

    if not os.path.exists(root_path):
        raise FileNotFoundError(f"Folder '{root_path}' not found.")

    X_list = []
    y_list = []
    subject_list = []
    session_list = [] 

    sessions_ids = [1, 2, 3]

    for session_id in sessions_ids:
        session_path = os.path.join(root_path, str(session_id))
        if not os.path.exists(session_path):
            print(f"Warning: Session folder {session_path} not found. Skipping.")
            continue

        print(f"Loading Session {session_id} from {session_path}...")

        current_labels = load_labels_from_file(root_path, session_id)
        if current_labels is None:
            print(f"  No label file found. Using OFFICIAL DOCS labels.")
            current_labels = SESSION_LABELS_FALLBACK.get(session_id)

        files = [f for f in os.listdir(session_path) if f.endswith('.mat')]
        try:
            files.sort(key=lambda x: int(x.split('_')[0]))
        except:
            files.sort()

        if len(files) == 0: continue

        for sub_idx, filename in enumerate(files):
            file_path = os.path.join(session_path, filename)
            try:
                mat_data = sio.loadmat(file_path)
            except:
                continue

            keys = [k for k in mat_data.keys() if k.startswith('de_') and not k.startswith('de_moving')]
            if not keys: keys = [k for k in mat_data.keys() if k.startswith('de_moving')]
            keys.sort(key=lambda x: int(''.join(filter(str.isdigit, x))))

            for trial_idx, key in enumerate(keys):
                if trial_idx >= 24: break

                trial_data = mat_data[key]
                if trial_data.shape[0] == 62:
                    trial_data = np.transpose(trial_data, (1, 0, 2))

                n_samples = trial_data.shape[0]
                trial_data_flat = trial_data.reshape(n_samples, -1)

                label = current_labels[trial_idx] if trial_idx < len(current_labels) else 0

                X_list.append(trial_data_flat)
                y_list.append(np.full(n_samples, label))
                subject_list.append(np.full(n_samples, sub_idx))
                session_list.append(np.full(n_samples, session_id)) # Track Session

    if not X_list: raise ValueError("No data loaded.")

    X_all = np.vstack(X_list)
    y_all = np.concatenate(y_list)
    subjects_all = np.concatenate(subject_list)
    sessions_all = np.concatenate(session_list)

    return X_all, y_all, subjects_all, sessions_all

try:
    X_all, y_all, subjects_all, sessions_all = load_seed4_from_folders(ROOT_PATH)
    print("\n--- Data Ready ---")
    print(f"Features: {X_all.shape}, Subjects: {np.unique(subjects_all)}, Sessions: {np.unique(sessions_all)}")
except Exception as e:
    print(f"Loading failed: {e}")
    X_all, y_all, subjects_all, sessions_all = None, None, None, None

## Step 4: Instance Selection Algorithm


In [ ]:
def select_instances(X_source, y_source, X_target_cal, y_target_cal, total_k=K_INSTANCES):

    # ALGORITHM STEPS:
    #1. Train Classifier C0 on Target Calibration Data (labeled).
    #2. Predict probabilities on Source Data.
    #3. Select top k Source instances per class that 'look most like' the Target.

    if len(np.unique(y_target_cal)) < 2:
        indices = np.random.choice(len(X_source), min(len(X_source), total_k), replace=False)
        return X_source[indices], y_source[indices]

    # RBF for instance selection
    c0 = SVC(kernel='rbf', probability=True, random_state=42, class_weight='balanced')
    c0.fit(X_target_cal, y_target_cal)

    class_map = {label: idx for idx, label in enumerate(c0.classes_)}

    try:
        probas = c0.predict_proba(X_source)
    except:
        dists = c0.decision_function(X_source)
        if dists.ndim == 1:
            probas = 1 / (1 + np.exp(-dists))
        else:
            probas = (dists - dists.min()) / (dists.max() - dists.min())

    selected_indices = []
    unique_classes = np.unique(y_source)
    n_classes = len(unique_classes)
    k_per_class = max(1, total_k // n_classes)

    for c in unique_classes:
        true_class_indices = np.where(y_source == c)[0]
        if len(true_class_indices) == 0: continue

        if c in class_map:
            col_idx = class_map[c]
            if probas.ndim == 2:
                class_scores = probas[true_class_indices, col_idx]
            else:
                if col_idx == 1:
                    class_scores = probas[true_class_indices]
                else:
                    class_scores = 1 - probas[true_class_indices]
        else:
            # Class c exists in source but NOT in target calibration =>
            # assign 0 score so these instances are unlikely to be picked
            class_scores = np.zeros(len(true_class_indices))

        sorted_local_indices = np.argsort(class_scores)[::-1]
        actual_k = min(len(sorted_local_indices), k_per_class)
        top_k_local = sorted_local_indices[:actual_k]
        selected_indices.extend(true_class_indices[top_k_local])

    selected_indices = np.array(selected_indices)
    return X_source[selected_indices], y_source[selected_indices]

## Step 5: Style Transfer Data Mapping Algorithm  

In [ ]:
def style_transfer_mapping(X_source_sel, y_source_sel, X_target_cal, y_target_cal, X_target_test):

    # ALGORITHM STEPS:
    #1. Calculate centroids (means) of each class for selected source and target calibration samples.
    #2. Project target calibration samples to 'O' space.
    #3. Find affine transformation (A, b) mapping O -> source means.
    #4. Apply A, b to X_target_test.

    # calculate class means
    classes = np.unique(y_target_cal)
    mu_source = {}
    mu_target = {}

    for c in classes:
        s_data = X_source_sel[y_source_sel == c]
        mu_source[c] = np.mean(s_data, axis=0) if len(s_data) > 0 else np.zeros(X_source_sel.shape[1])
        t_data = X_target_cal[y_target_cal == c]
        mu_target[c] = np.mean(t_data, axis=0) if len(t_data) > 0 else np.zeros(X_target_cal.shape[1])

    # construct 'O' (Projected Target Calibration)
    # o_i = mu_source + rho * (t_i - mu_target) (formula from Mind To Music paper)
    O_train = []
    S_train = []

    for i, t_sample in enumerate(X_target_cal):
        label = y_target_cal[i]
        o_i = mu_source[label] + RHO * (t_sample - mu_target[label])
        O_train.append(o_i)
        S_train.append(mu_source[label]) # map to source mean

    O_train = np.array(O_train)
    S_train = np.array(S_train)

    # project the test target data using Ridge Regression
    ridge = Ridge(alpha=1.0)
    ridge.fit(O_train, S_train)
    X_test_mapped = ridge.predict(X_target_test)

    return X_test_mapped


## Step 6: Execution and Evaluation

In [ ]:
def get_balanced_calibration_split(X, y, cal_ratio=0.2):
    # Incorporates a calibration phase where we ensure we capture
    # some data for EVERY class.

    cal_indices = []
    test_indices = []

    unique_classes = np.unique(y)

    for c in unique_classes:
        class_indices = np.where(y == c)[0]

        n_cal = int(len(class_indices) * cal_ratio)
        if n_cal < 5 and len(class_indices) >= 5: n_cal = 5 #  min 5 samples

        cal_indices.extend(class_indices[:n_cal])
        test_indices.extend(class_indices[n_cal:])

    return X[cal_indices], y[cal_indices], X[test_indices], y[test_indices]

def run_validation(X, y, subjects, sessions):
    if X is None: return

    unique_subjects = np.unique(subjects)
    unique_sessions = np.unique(sessions)

    sstm_accuracies = []
    baseline_accuracies = []
    fusion_accuracies = []

    print(f"\n--- Starting Per-Session Validation ({len(unique_subjects)} Subjects, {len(unique_sessions)} Sessions) ---")

    for target_subj in unique_subjects:
        for sess_id in unique_sessions:
            print(f"\nProcessing Subject {target_subj} | Session {sess_id}...")

            source_mask = (subjects != target_subj)
            target_mask = (subjects == target_subj) & (sessions == sess_id)

            if np.sum(target_mask) == 0: continue

            X_source = X[source_mask]
            y_source = y[source_mask]
            X_target = X[target_mask]
            y_target = y[target_mask]

            scaler_source = StandardScaler()
            X_source = scaler_source.fit_transform(X_source)

            scaler_target = StandardScaler()
            X_target_scaled = scaler_target.fit_transform(X_target)

            X_cal, y_cal, X_test, y_test = get_balanced_calibration_split(X_target_scaled, y_target, cal_ratio=0.2)

            # BASELINE (Probabilistic)
            clf_base = SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced', probability=True)
            idx = np.random.choice(len(X_source), size=min(len(X_source), K_INSTANCES), replace=False)
            clf_base.fit(X_source[idx], y_source[idx])

            probs_base = clf_base.predict_proba(X_test)
            base_acc = accuracy_score(y_test, np.argmax(probs_base, axis=1))
            baseline_accuracies.append(base_acc)

            # SSTM (Probabilistic)
            # Instance Selection
            X_source_sel, y_source_sel = select_instances(X_source, y_source, X_cal, y_cal, total_k=K_INSTANCES)
            # Mapping
            X_test_mapped = style_transfer_mapping(X_source_sel, y_source_sel, X_cal, y_cal, X_test)

            # Classification
            clf_sstm = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True)
            clf_sstm.fit(X_source_sel, y_source_sel)

            probs_sstm = clf_sstm.predict_proba(X_test_mapped)
            sstm_acc = accuracy_score(y_test, np.argmax(probs_sstm, axis=1))
            sstm_accuracies.append(sstm_acc)

            # FUSION
            # Average the confidence scores from both models
            probs_fusion = (probs_base + probs_sstm) / 2
            fusion_pred = np.argmax(probs_fusion, axis=1)
            fusion_acc = accuracy_score(y_test, fusion_pred)
            fusion_accuracies.append(fusion_acc)

            print(f"   Base: {base_acc*100:.1f}% | SSTM: {sstm_acc*100:.1f}% | Fusion: {fusion_acc*100:.1f}%")

    print("\n==========================================")
    print(f"Avg Baseline: {np.mean(baseline_accuracies)*100:.2f}%")
    print(f"Avg SSTM:     {np.mean(sstm_accuracies)*100:.2f}%")
    print(f"Avg Fusion:   {np.mean(fusion_accuracies)*100:.2f}%")
    print("==========================================")

if __name__ == "__main__":
    if X_all is not None:
        run_validation(X_all, y_all, subjects_all, sessions_all)